In [ ]:
!pip install imbalanced-learn
!pip install pytorch_tabular pytorch_tabnet xgboost lightgbm catboost imbalanced-learn
!pip install scikit-posthocs statsmodels

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================================
# 1. Install Kaggle API
# ==========================================================
!pip install -q kaggle

# ==========================================================
# 2. Download NASA Software Defect Prediction Dataset
# ==========================================================
!kaggle datasets download -d aczy156/software-defect-prediction-nasa

# ==========================================================
# 3. Unzip Dataset
# ==========================================================
!unzip -q software-defect-prediction-nasa.zip -d nasa_defect_data

# ==========================================================
# 4. Check Dataset Structure
# ==========================================================
import os

print("Top-level files/folders:")
print(os.listdir("nasa_defect_data"))

print("\nFull directory structure:")
for root, dirs, files in os.walk("nasa_defect_data"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# ==========================================================
# 1. Install Kaggle API
# ==========================================================
!pip install -q kaggle

# ==========================================================
# 2. Download AEEEM Software Defect Prediction Dataset
# ==========================================================
!kaggle datasets download -d faisalmohammad/aeeem-dataet4sdp

# ==========================================================
# 3. Unzip Dataset
# ==========================================================
!unzip -q aeeem-dataet4sdp.zip -d aeeem_data

# ==========================================================
# 4. Check Dataset Structure
# ==========================================================
import os

print("Top-level files/folders:")
print(os.listdir("aeeem_data"))

print("\nFull directory structure:")
for root, dirs, files in os.walk("aeeem_data"):
    for file in files:
        print(os.path.join(root, file))

Dataset URL: https://www.kaggle.com/datasets/faisalmohammad/aeeem-dataet4sdp
License(s): apache-2.0
100% 364k/364k [00:00<00:00, 34.9MB/s]

Top-level files/folders:
['EQ.csv', 'JDT.csv', 'PDE.csv', 'ML.csv', 'LC.csv']

Full directory structure:
aeeem_data/EQ.csv
aeeem_data/JDT.csv
aeeem_data/PDE.csv
aeeem_data/ML.csv
aeeem_data/LC.csv


In [ ]:
###########################################
            #Data Harmonization
###########################################

import pandas as pd
import os
from glob import glob
import re

input_folder = "aeeem_data"
output_folder = "aeeem_data_arff"

os.makedirs(output_folder, exist_ok=True)

for csv_file in glob(os.path.join(input_folder, "*.csv")):

    df = pd.read_csv(csv_file)
    df.columns = [
        re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        for col in df.columns
        ]
    target_col = "class"

    df["Defective"] = (
        df[target_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "buggy": "Y",
            "clean": "N"
        })
    )

    df = df.drop(columns=[target_col])

    cols = [c for c in df.columns if c != "Defective"] + ["Defective"]
    df = df[cols]

    relation_name = os.path.splitext(os.path.basename(csv_file))[0]
    arff_file = os.path.join(output_folder, relation_name + ".arff")

    with open(arff_file, "w", encoding="utf-8") as f:

        f.write(f"@relation {relation_name}\n\n")

        for col in df.columns[:-1]:
            f.write(f"@attribute {col} numeric\n")

        f.write("@attribute Defective {Y,N}\n\n")

        f.write("@data\n")

        for _, row in df.iterrows():
            values = [
                "?" if pd.isna(v) else str(v)
                for v in row
            ]
            f.write(",".join(values) + "\n")

    print(f"Converted: {csv_file} -> {arff_file}")

print("Done.")

Converted: aeeem_data/EQ.csv -> aeeem_data_arff/EQ.arff
Converted: aeeem_data/JDT.csv -> aeeem_data_arff/JDT.arff
Converted: aeeem_data/PDE.csv -> aeeem_data_arff/PDE.arff
Converted: aeeem_data/ML.csv -> aeeem_data_arff/ML.arff
Converted: aeeem_data/LC.csv -> aeeem_data_arff/LC.arff
Done.


In [ ]:
import os
import shutil

# Source folders
source_dirs = ["nasa_defect_data", "aeeem_data_arff"]

# Destination folder
dest_dir = "defect_data"

if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
    print(f"Deleted folder: {dest_dir}")
else:
    print(f"Folder not found: {dest_dir}")


# Create destination folder if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Copy all .arff files
for source_dir in source_dirs:
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.lower().endswith(".arff"):
                src_path = os.path.join(root, file)
                dst_path = os.path.join(dest_dir, file)

                # Handle duplicate filenames
                if os.path.exists(dst_path):
                    base, ext = os.path.splitext(file)
                    counter = 1
                    while os.path.exists(dst_path):
                        dst_path = os.path.join(
                            dest_dir, f"{base}_{counter}{ext}"
                        )
                        counter += 1

                shutil.copy2(src_path, dst_path)
                print(f"Copied: {src_path} -> {dst_path}")

print("Done!")

Folder not found: defect_data
Copied: nasa_defect_data/CM1.arff -> defect_data/CM1.arff
Copied: nasa_defect_data/PC5.arff -> defect_data/PC5.arff
Copied: nasa_defect_data/JM1.arff -> defect_data/JM1.arff
Copied: nasa_defect_data/PC2.arff -> defect_data/PC2.arff
Copied: nasa_defect_data/MC2.arff -> defect_data/MC2.arff
Copied: nasa_defect_data/KC4.arff -> defect_data/KC4.arff
Copied: nasa_defect_data/PC1.arff -> defect_data/PC1.arff
Copied: nasa_defect_data/PC3.arff -> defect_data/PC3.arff
Copied: nasa_defect_data/KC3.arff -> defect_data/KC3.arff
Copied: nasa_defect_data/PC4.arff -> defect_data/PC4.arff
Copied: nasa_defect_data/MW1.arff -> defect_data/MW1.arff
Copied: nasa_defect_data/KC1.arff -> defect_data/KC1.arff
Copied: nasa_defect_data/MC1.arff -> defect_data/MC1.arff
Copied: aeeem_data_arff/ML.arff -> defect_data/ML.arff
Copied: aeeem_data_arff/EQ.arff -> defect_data/EQ.arff
Copied: aeeem_data_arff/PDE.arff -> defect_data/PDE.arff
Copied: aeeem_data_arff/JDT.arff -> defect_data/J

In [ ]:
import os
import numpy as np
import pandas as pd
import shap

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler


# ============================================================
# FOLD-LEVEL FEATURE SELECTION
# ============================================================

def select_features_fold(
    X_train,
    y_train,
    dataset_name,
    fold_idx,
    output_root=None
):
    """
    Perform RF + SHAP feature selection using ONLY the
    training data of the current CV fold.

    Generates:
        1. RF feature importance bar plot
        2. SHAP beeswarm summary plot
        3. Mean |SHAP| bar plot
        4. RF + SHAP combined heatmap
        5. Feature importance CSV
        6. Selected feature TXT file

    Returns
    -------
    selected_features : list
        Selected feature names for this fold.
    """

    # --------------------------------------------------------
    # Copy data
    # --------------------------------------------------------

    X_train = X_train.copy()
    y_train = pd.Series(y_train).copy()

    # --------------------------------------------------------
    # Make sure all features are numeric
    # --------------------------------------------------------

    X_train = X_train.apply(
        pd.to_numeric,
        errors="coerce"
    )

    # Remove columns that are completely NaN
    X_train = X_train.dropna(
        axis=1,
        how="all"
    )

    # --------------------------------------------------------
    # TRAINING-FOLD-ONLY IMPUTATION
    # --------------------------------------------------------

    train_medians = X_train.median()

    X_train = X_train.fillna(
        train_medians
    )

    # If a column is still NaN
    X_train = X_train.fillna(0)

    # --------------------------------------------------------
    # Check target
    # --------------------------------------------------------

    if y_train.nunique() < 2:
        raise ValueError(
            f"{dataset_name} - Fold {fold_idx}: "
            "Training set contains fewer than two classes."
        )

    # --------------------------------------------------------
    # Standardization
    #
    # Fitted ONLY on current training fold
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_train_scaled = pd.DataFrame(
        X_train_scaled,
        columns=X_train.columns,
        index=X_train.index
    )

    # ========================================================
    # RANDOM FOREST FEATURE IMPORTANCE
    # ========================================================

    print(
        f"    [Feature Selection] "
        f"Fitting Random Forest for Fold {fold_idx}..."
    )

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf.fit(
        X_train_scaled,
        y_train
    )

    rf_importance = pd.Series(
        rf.feature_importances_,
        index=X_train.columns,
        name="RF_Importance"
    ).sort_values(
        ascending=False
    )

    # --------------------------------------------------------
    # RF threshold = 50th percentile
    # --------------------------------------------------------

    rf_threshold = np.percentile(
        rf_importance.values,
        50
    )

    rf_selected = rf_importance[
        rf_importance >= rf_threshold
    ].index.tolist()

    # ========================================================
    # SHAP FEATURE IMPORTANCE
    # ========================================================

    print(
        f"    [Feature Selection] "
        f"Computing SHAP for Fold {fold_idx}..."
    )

    # --------------------------------------------------------
    # SHAP background
    # ONLY training-fold observations
    # --------------------------------------------------------

    background_size = min(
        100,
        len(X_train_scaled)
    )

    if len(X_train_scaled) > background_size:

        background = X_train_scaled.sample(
            n=background_size,
            random_state=42
        )

    else:

        background = X_train_scaled

    # --------------------------------------------------------
    # SHAP TreeExplainer
    # --------------------------------------------------------

    explainer = shap.TreeExplainer(
        rf,
        data=background
    )

    shap_values = explainer.shap_values(
        X_train_scaled,
        check_additivity=False
    )

    # --------------------------------------------------------
    # Handle different SHAP output formats
    # --------------------------------------------------------

    if isinstance(shap_values, list):

        if len(shap_values) == 2:

            # Positive class
            shap_array = shap_values[1]

        else:

            shap_array = shap_values[0]

    else:

        shap_array = np.asarray(
            shap_values
        )

        # Newer SHAP versions:
        # (samples, features, classes)

        if shap_array.ndim == 3:

            if shap_array.shape[2] == 2:

                # Positive class
                shap_array = shap_array[:, :, 1]

            else:

                shap_array = shap_array[:, :, 0]

        elif shap_array.ndim != 2:

            raise ValueError(
                f"Unexpected SHAP output shape: "
                f"{shap_array.shape}"
            )

    # --------------------------------------------------------
    # Mean absolute SHAP importance
    # --------------------------------------------------------

    shap_importance = pd.Series(
        np.mean(
            np.abs(shap_array),
            axis=0
        ),
        index=X_train.columns,
        name="SHAP_Importance"
    ).sort_values(
        ascending=False
    )

    # --------------------------------------------------------
    # SHAP threshold = 50th percentile
    # --------------------------------------------------------

    shap_threshold = np.percentile(
        shap_importance.values,
        50
    )

    shap_selected = shap_importance[
        shap_importance >= shap_threshold
    ].index.tolist()

    # ========================================================
    # CONSENSUS SELECTION
    #
    # RF AND SHAP must both select the feature
    # ========================================================

    consensus_selected = [
        feature
        for feature in X_train.columns
        if feature in rf_selected
        and feature in shap_selected
    ]

    # --------------------------------------------------------
    # Safety fallback
    # --------------------------------------------------------

    if len(consensus_selected) == 0:

        combined_importance = (
            rf_importance.rank(pct=True)
            + shap_importance.rank(pct=True)
        )

        best_feature = combined_importance.idxmax()

        consensus_selected = [
            best_feature
        ]

    # ========================================================
    # SAVE RESULTS + FIGURES
    # ========================================================

    if output_root is not None:

        # ----------------------------------------------------
        # IMPORTANT:
        # Separate directory for each fold
        # ----------------------------------------------------

        fold_output_dir = os.path.join(
            output_root,
            "feature_selection",
            f"fold_{fold_idx}"
        )

        os.makedirs(
            fold_output_dir,
            exist_ok=True
        )

        # ====================================================
        # 1. RANDOM FOREST FEATURE IMPORTANCE PLOT
        # ====================================================

        rf_plot_values = rf_importance.sort_values(
            ascending=True
        )

        n_features = len(
            rf_plot_values
        )

        fig_height = max(
            5,
            n_features * 0.28
        )

        fig, ax = plt.subplots(
            figsize=(10, fig_height)
        )

        colors = sns.color_palette(
            "GnBu",
            n_colors=n_features
        )

        bars = ax.barh(
            rf_plot_values.index,
            rf_plot_values.values,
            color=colors,
            edgecolor="none"
        )

        # Highlight selected features
        for bar, feature in zip(
            bars,
            rf_plot_values.index
        ):

            if feature in rf_selected:

                bar.set_edgecolor(
                    "#2c2c2c"
                )

                bar.set_linewidth(
                    0.8
                )

        ax.axvline(
            rf_threshold,
            color="#e05c5c",
            linewidth=1.4,
            linestyle="--",
            label=(
                f"p50 threshold = "
                f"{rf_threshold:.4f}"
            )
        )

        ax.set_xlabel(
            "Random Forest Feature Importance",
            fontsize=11
        )

        ax.set_ylabel(
            "Feature",
            fontsize=11
        )

        ax.set_title(
            f"RF Feature Importance — "
            f"{dataset_name} — Fold {fold_idx}",
            fontsize=13,
            fontweight="bold"
        )

        ax.tick_params(
            axis="y",
            labelsize=8
        )

        ax.legend(
            fontsize=9
        )

        sns.despine(
            ax=ax
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                fold_output_dir,
                "rf_importance.png"
            ),
            dpi=150,
            bbox_inches="tight"
        )

        plt.close()

        # ====================================================
        # 2. SHAP BEESWARM SUMMARY PLOT
        # ====================================================

        plt.figure(
            figsize=(
                10,
                max(
                    5,
                    n_features * 0.28
                )
            )
        )

        shap.summary_plot(
            shap_array,
            X_train_scaled,
            feature_names=X_train.columns,
            show=False,
            plot_size=None,
            color_bar=True
        )

        plt.title(
            f"SHAP Summary (Beeswarm) — "
            f"{dataset_name} — Fold {fold_idx}",
            fontsize=12,
            fontweight="bold",
            pad=10
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                fold_output_dir,
                "shap_beeswarm.png"
            ),
            dpi=150,
            bbox_inches="tight"
        )

        plt.close()

        # ====================================================
        # 3. MEAN |SHAP| BAR PLOT
        # ====================================================

        shap_plot_values = shap_importance.sort_values(
            ascending=True
        )

        fig_height = max(
            5,
            n_features * 0.28
        )

        fig, ax = plt.subplots(
            figsize=(10, fig_height)
        )

        colors = sns.color_palette(
            "OrRd",
            n_colors=n_features
        )

        bars = ax.barh(
            shap_plot_values.index,
            shap_plot_values.values,
            color=colors,
            edgecolor="none"
        )

        # Highlight selected features
        for bar, feature in zip(
            bars,
            shap_plot_values.index
        ):

            if feature in shap_selected:

                bar.set_edgecolor(
                    "#2c2c2c"
                )

                bar.set_linewidth(
                    0.8
                )

        ax.axvline(
            shap_threshold,
            color="#e05c5c",
            linewidth=1.4,
            linestyle="--",
            label=(
                f"p50 threshold = "
                f"{shap_threshold:.4f}"
            )
        )

        ax.set_xlabel(
            "Mean |SHAP value|",
            fontsize=11
        )

        ax.set_ylabel(
            "Feature",
            fontsize=11
        )

        ax.set_title(
            f"Mean |SHAP| — "
            f"{dataset_name} — Fold {fold_idx}",
            fontsize=13,
            fontweight="bold"
        )

        ax.tick_params(
            axis="y",
            labelsize=8
        )

        ax.legend(
            fontsize=9
        )

        sns.despine(
            ax=ax
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                fold_output_dir,
                "shap_bar.png"
            ),
            dpi=150,
            bbox_inches="tight"
        )

        plt.close()

        # ====================================================
        # 4. COMBINED RF + SHAP HEATMAP
        # ====================================================

        all_features = (
            rf_importance.index
            .union(
                shap_importance.index
            )
        )

        rf_values = rf_importance.reindex(
            all_features,
            fill_value=0
        )

        shap_values_series = shap_importance.reindex(
            all_features,
            fill_value=0
        )

        # Normalize RF importance
        if rf_values.max() > 0:

            rf_norm = (
                rf_values /
                rf_values.max()
            )

        else:

            rf_norm = rf_values

        # Normalize SHAP importance
        if shap_values_series.max() > 0:

            shap_norm = (
                shap_values_series /
                shap_values_series.max()
            )

        else:

            shap_norm = shap_values_series

        # Average normalized importance
        avg_score = (
            rf_norm +
            shap_norm
        ) / 2

        order = avg_score.sort_values(
            ascending=False
        ).index

        heat_df = pd.DataFrame({
            "RF Importance\n(normalised)": (
                rf_norm.loc[order]
            ),
            "Mean |SHAP|\n(normalised)": (
                shap_norm.loc[order]
            )
        })

        fig_height = max(
            5,
            len(order) * 0.30
        )

        fig, ax = plt.subplots(
            figsize=(7, fig_height)
        )

        sns.heatmap(
            heat_df,
            annot=True,
            fmt=".2f",
            cmap="YlOrRd",
            linewidths=0.4,
            linecolor="#cccccc",
            ax=ax,
            cbar_kws={
                "shrink": 0.6
            }
        )

        ax.set_title(
            f"Combined Importance Heatmap — "
            f"{dataset_name} — Fold {fold_idx}",
            fontsize=12,
            fontweight="bold",
            pad=10
        )

        ax.tick_params(
            axis="y",
            labelsize=8,
            rotation=0
        )

        ax.tick_params(
            axis="x",
            labelsize=9
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                fold_output_dir,
                "combined_importance_heatmap.png"
            ),
            dpi=150,
            bbox_inches="tight"
        )

        plt.close()

        # ====================================================
        # 5. SAVE IMPORTANCE SCORES CSV
        # ====================================================

        scores_df = pd.DataFrame({

            "Feature": X_train.columns,

            "RF_Importance": [
                rf_importance[f]
                for f in X_train.columns
            ],

            "SHAP_Importance": [
                shap_importance[f]
                for f in X_train.columns
            ],

            "RF_Selected": [
                f in rf_selected
                for f in X_train.columns
            ],

            "SHAP_Selected": [
                f in shap_selected
                for f in X_train.columns
            ],

            "Consensus_Selected": [
                f in consensus_selected
                for f in X_train.columns
            ]
        })

        scores_df = scores_df.sort_values(
            "SHAP_Importance",
            ascending=False
        )

        scores_df.to_csv(
            os.path.join(
                fold_output_dir,
                "feature_importance.csv"
            ),
            index=False
        )

        # ====================================================
        # 6. SAVE SELECTED FEATURES
        # ====================================================

        with open(
            os.path.join(
                fold_output_dir,
                "selected_features.txt"
            ),
            "w"
        ) as f:

            for feature in consensus_selected:

                f.write(
                    feature + "\n"
                )

    # ========================================================
    # PRINT RESULT
    # ========================================================

    print(
        f"\n{dataset_name} - Fold {fold_idx}"
    )

    print(
        f"Total features : "
        f"{len(X_train.columns)}"
    )

    print(
        f"RF selected    : "
        f"{len(rf_selected)}"
    )

    print(
        f"SHAP selected  : "
        f"{len(shap_selected)}"
    )

    print(
        f"Final selected : "
        f"{len(consensus_selected)}"
    )

    print(
        "Selected features:"
    )

    print(
        consensus_selected
    )

    # ========================================================
    # RETURN FEATURE LIST
    # ========================================================

    return consensus_selected

In [ ]:
import os
import json
import time
import traceback
import warnings
warnings.filterwarnings("ignore")
import shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import arff
from scipy.stats import friedmanchisquare

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from imblearn.over_sampling import BorderlineSMOTE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve, auc
)
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ---- Mixed-effects modeling (dataset as random effect) ----
try:
    import statsmodels.formula.api as smf
    MIXEDLM_AVAILABLE = True
except ImportError:
    MIXEDLM_AVAILABLE = False
    print("statsmodels not installed. Run: pip install statsmodels")


def zip_dataset_results(out_dir, dataset_name):
    """Zip out_dir into OUTPUT_ROOT/dataset_name.zip, then leave the folder as-is."""
    zip_path = os.path.join(OUTPUT_ROOT, dataset_name)  # shutil adds .zip automatically
    shutil.make_archive(zip_path, "zip", root_dir=out_dir)
    print(f"  [Zip] Created {zip_path}.zip")


def is_dataset_already_done(dataset_name):
    """Resume support: if the zip already exists, this dataset is considered complete."""
    return os.path.exists(os.path.join(OUTPUT_ROOT, f"{dataset_name}.zip"))



# ---- PyTorch / weights_only safe-globals fix (PyTorch >= 2.6) ----
import torch
try:
    import omegaconf
    import omegaconf.dictconfig
    import omegaconf.listconfig
    import omegaconf.base
    import omegaconf.nodes
    torch.serialization.add_safe_globals([
        omegaconf.dictconfig.DictConfig,
        omegaconf.listconfig.ListConfig,
        omegaconf.base.ContainerMetadata,
        omegaconf.base.Metadata,
        omegaconf.nodes.AnyNode,
        omegaconf.nodes.IntegerNode,
        omegaconf.nodes.FloatNode,
        omegaconf.nodes.StringNode,
        omegaconf.nodes.BooleanNode,
        omegaconf.nodes.EnumNode,
        omegaconf.nodes.InterpolationResultNode,
    ])
except Exception:
    pass

# ---- Timing / GPU-memory measurement helpers (Colab T4) ----
GPU_AVAILABLE = torch.cuda.is_available()
if GPU_AVAILABLE:
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected -- GPU memory columns will read 0.0 for every model.")




def _gpu_reset():
    """Call right before a train/inference block to zero out the peak-memory counter."""
    if GPU_AVAILABLE:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()


def _gpu_peak_mb():
    """Peak GPU memory (MB) allocated since the last _gpu_reset() call.
    Sklearn/XGBoost/LightGBM/CatBoost (CPU-bound here) will report ~0."""
    if GPU_AVAILABLE:
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0.0


class _TimedBlock:
    """Context manager: measures wall-clock time + GPU peak memory for one block.
    Usage:
        with _TimedBlock() as t:
            model.fit(...)
        t.seconds, t.gpu_mb
    """
    def __enter__(self):
        _gpu_reset()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.seconds = time.perf_counter() - self._t0
        self.gpu_mb = _gpu_peak_mb()
        return False


from pytorch_tabnet.tab_model import TabNetClassifier

from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig
from pytorch_tabular.models.ft_transformer.config import FTTransformerConfig
from pytorch_tabular.models.tab_transformer.config import TabTransformerConfig
from pytorch_tabular.models.autoint.config import AutoIntConfig
from pytorch_tabular.models.gandalf.config import GANDALFConfig
from pytorch_tabular.models.danet.config import DANetConfig

# NOTE: NODE model removed per request (import, config, and usage all dropped).

# ==========================================================
# CONFIG
# ==========================================================

DATA_DIR = "/content/defect_data"

DATASET_FILES = [
    "PC1.arff", "PC2.arff", "PC3.arff", "PC4.arff", "PC5.arff",
    "JM1.arff", "KC1.arff", "KC3.arff",
    "CM1.arff", "MC1.arff", "MC2.arff", "MW1.arff",
    "EQ.arff","JDT.arff","LC.arff","ML.arff","PDE.arff"
]





RANDOM_STATE = 42
N_SPLITS = 5  # k-fold CV

# ---- Deduplication ----
# Exact duplicate rows (features + label) are dropped BEFORE the CV split so
# copies of the same instance can never sit in both train and test folds.
DEDUPLICATE = True

# ---- Meta-Stacking (leak-free) ----
# The meta-model is trained ONLY on out-of-fold predictions generated inside the
# TRAIN fold (inner CV). The test fold is never used to fit anything.
# Cost: every base model is refit META_INNER_SPLITS extra times per outer fold.
META_INNER_SPLITS = 3       # 3 = cheaper, 5 = more accurate but ~5x extra cost
META_BASE_MODELS = None     # None = all pure models; or e.g. ["RandomForest","XGBoost","CatBoost","TabNet"]

# ---- Execution Mode ----
# 1 = No SMOTE, all features
# 2 = With SMOTE (BorderlineSMOTE), all features
# 3 = No SMOTE, pre-selected features per dataset  [current]
# 4 = With SMOTE, pre-selected features per dataset
EXECUTION_MODE = 4


MODE_SUFFIX = {
    1: "results_no_smote",
    2: "results_smote",
    3: "results_no_smote_feature_selection",
    4: "results_smote_feature_selection",
}

OUTPUT_ROOT = f"/content/drive/MyDrive/Software_Defects_Revised/{MODE_SUFFIX[EXECUTION_MODE]}"





TARGET_CANDIDATES = [
    "Defective", "defects", "Defects", "DEFECT", "problems",
    "class", "Class", "label", "Label"
]

MODEL_ORDER = [
    # Classical ML Models
    "LogisticRegression", "NaiveBayes", "RandomForest", "ExtraTrees",
    "XGBoost", "LightGBM", "CatBoost",
    # Deep Tabular Models
    "TabNet", "GANDALF", "DANet",
    # Transformer-Based Models
    "FTTransformer", "TabTransformer", "AutoInt",
    # Ensembles
    "Soft Voting", "Hard Voting", "Stacking (sklearn)", "Meta-Stacking",
]

MODEL_FAMILY = {
    "LogisticRegression": "Classical", "NaiveBayes": "Classical",
    "RandomForest": "Classical", "ExtraTrees": "Classical",
    "XGBoost": "Classical", "LightGBM": "Classical", "CatBoost": "Classical",
    "TabNet": "DeepTabular", "GANDALF": "DeepTabular", "DANet": "DeepTabular",
    "FTTransformer": "Transformer", "TabTransformer": "Transformer",
    "AutoInt": "Transformer",
    "Soft Voting": "Ensemble", "Hard Voting": "Ensemble",
    "Stacking (sklearn)": "Ensemble", "Meta-Stacking": "Ensemble",
}

# Models that are NOT ensembles -- i.e. the "pure" base learners.
# Used to (a) restrict Meta-Stacking's inputs to base-model probabilities only
# (so it never averages Soft/Hard Voting or sklearn Stacking outputs),
# and (b) define which families participate in the statistical-validity tests.
PURE_MODEL_NAMES = {
    name for name, fam in MODEL_FAMILY.items() if fam != "Ensemble"
}

# "Ensemble" is intentionally NOT a model family for statistical-validity
# purposes -- it's a combination of the other families, not a comparable
# treatment in its own right. Only these three are real families.
STAT_FAMILIES = ["Classical", "DeepTabular", "Transformer"]

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Collects fold-level metrics across ALL datasets for the final
# cross-dataset statistical tests (Friedman/Nemenyi + Mixed Effects).
GLOBAL_FOLD_RECORDS = []

def load_global_records_for_done_dataset(dataset_name, out_dir):
    """
    Resume support: if a dataset was already completed (zipped), reload its
    foldwise metrics CSV from disk so it still contributes to the
    cross-dataset (global) statistical tests. Without this, resuming a run
    silently drops every previously-completed dataset from the Friedman /
    Mixed-Effects global comparison.
    """
    csv_path = os.path.join(out_dir, f"{dataset_name}_foldwise_metrics.csv")

    if not os.path.exists(csv_path):
        # Fallback: folder was removed/archived -- try pulling the CSV out of the zip.
        zip_path = os.path.join(OUTPUT_ROOT, f"{dataset_name}.zip")
        if os.path.exists(zip_path):
            import zipfile
            try:
                with zipfile.ZipFile(zip_path, "r") as zf:
                    inner_name = f"{dataset_name}_foldwise_metrics.csv"
                    matches = [n for n in zf.namelist() if n.endswith(inner_name)]
                    if matches:
                        with zf.open(matches[0]) as f:
                            df = pd.read_csv(f)
                            return df.to_dict("records")
            except Exception as e:
                print(f"  !! Could not read foldwise CSV from zip for {dataset_name}: {e}")
        print(f"  !! No foldwise metrics found on disk or in zip for {dataset_name} "
              f"-- it will be MISSING from global statistical tests.")
        return []

    df = pd.read_csv(csv_path)
    return df.to_dict("records")



# ==========================================================
# DATA LOADING HELPERS
# ==========================================================

def load_dataset(path):
    if path.lower().endswith(".arff"):
        data, meta = arff.loadarff(path)
        df = pd.DataFrame(data)
    elif path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    for col in df.columns:
        if df[col].dtype == object:
            try:
                df[col] = df[col].apply(
                    lambda x: x.decode("utf-8") if isinstance(x, bytes) else x
                )
            except Exception:
                pass

    target_col = None
    for cand in TARGET_CANDIDATES:
        if cand in df.columns:
            target_col = cand
            break
    if target_col is None:
        target_col = df.columns[-1]

    df[target_col] = df[target_col].apply(_map_target_value)
    df = df.rename(columns={target_col: "target"})
    df = df.dropna(subset=["target"])
    df["target"] = df["target"].astype(int)

    for col in df.columns:
        if col == "target":
            continue
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(axis=1, how="all")
    #feature_cols = [c for c in df.columns if c != "target"]
    #df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

    # Drop exact duplicate rows BEFORE any CV split (prevents train/test copies).
    if DEDUPLICATE:
        n_before = len(df)
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"  [Dedup] {os.path.basename(path)}: {n_before} -> {len(df)} rows "
              f"({n_before - len(df)} exact duplicates removed)")

    return df


def _map_target_value(v):
    if isinstance(v, bytes):
        v = v.decode("utf-8")
    if isinstance(v, str):
        v_low = v.strip().lower()
        if v_low in ("y", "yes", "true", "1", "defective", "true_y"):
            return 1
        if v_low in ("n", "no", "false", "0", "not_defective", "false_y"):
            return 0
        try:
            return float(v_low)
        except ValueError:
            return np.nan
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan





# ==========================================================
# METRIC / SAVING HELPERS
# ==========================================================

def compute_metrics(y_true, y_pred, y_prob=None):
    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            out["ROC_AUC"] = roc_auc_score(y_true, y_prob)
        except Exception:
            out["ROC_AUC"] = np.nan
    else:
        out["ROC_AUC"] = np.nan
    return out


def log_error(errors_log_path, model_name, exc):
    with open(errors_log_path, "a") as f:
        f.write(f"\n[{model_name}] FAILED\n")
        f.write(traceback.format_exc())
        f.write("\n" + "-" * 60 + "\n")


def save_fold_artifacts(model_dir, model_name, fold_idx, y_test, y_pred, y_prob=None):
    """Save per-fold predictions / confusion matrix (lightweight, one CM per fold)."""
    os.makedirs(model_dir, exist_ok=True)
    pred_df = pd.DataFrame({
        "y_true": np.asarray(y_test).ravel(),
        "y_pred": np.asarray(y_pred).ravel(),
    })
    if y_prob is not None:
        pred_df["y_prob"] = np.asarray(y_prob).ravel()
    pred_df.to_csv(os.path.join(model_dir, f"fold{fold_idx}_predictions.csv"), index=False)

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="GnBu")
    plt.title(f"{model_name} — Fold {fold_idx}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(model_dir, f"fold{fold_idx}_confusion_matrix.png"), dpi=120)
    plt.close()


def aggregate_fold_metrics(fold_metrics_list):
    """Turn a list of per-fold metric dicts into mean +/- std summary dict."""
    df = pd.DataFrame(fold_metrics_list)
    summary = {}
    for col in ["Accuracy", "Precision", "Recall", "F1", "MCC", "ROC_AUC",
                "train_time_sec", "infer_time_sec", "train_gpu_mb", "infer_gpu_mb"]:
        if col in df.columns:
            summary[f"{col}_mean"] = df[col].mean()
            summary[f"{col}_std"] = df[col].std()
    return summary


# ==========================================================
# PER-DATASET PIPELINE (K-FOLD CV)
# ==========================================================

def run_pipeline_for_dataset(filename):
    dataset_name = os.path.splitext(os.path.basename(filename))[0]
    out_dir = os.path.join(OUTPUT_ROOT, dataset_name)
    dataset_path = os.path.join(DATA_DIR, filename)

    if is_dataset_already_done(dataset_name):
        print(f"\n[SKIP] {dataset_name} already zipped — reloading its fold records "
              f"for global stats and resuming from next dataset.")
        records = load_global_records_for_done_dataset(dataset_name, out_dir)
        GLOBAL_FOLD_RECORDS.extend(records)
        print(f"  [SKIP] Recovered {len(records)} fold-level rows for {dataset_name}.")
        return

    os.makedirs(out_dir, exist_ok=True)
    errors_log_path = os.path.join(out_dir, "errors.log")
    open(errors_log_path, "w").close()

    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    try:
        df = load_dataset(dataset_path)
    except Exception as e:
        log_error(errors_log_path, "DATA_LOADING", e)
        print(f"  !! Could not load {dataset_name}, skipping entirely.")
        return

    if df["target"].nunique() < 2:
        log_error(errors_log_path, "DATA_LOADING",
                   Exception("Target column has fewer than 2 classes."))
        print(f"  !! {dataset_name} has a single-class target, skipping.")
        return

    # ---- EDA plots (dataset-level, computed once) ----
    try:
        plt.figure(figsize=(6, 4))
        sns.countplot(x="target", data=df)
        plt.title(f"Class Distribution — {dataset_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "class_distribution.png"), dpi=150)
        plt.close()

        plt.figure(figsize=(14, 10))
        sns.heatmap(df.corr(), cmap="coolwarm", center=0)
        plt.title(f"Feature Correlation Heatmap — {dataset_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "correlation_heatmap.png"), dpi=150)
        plt.close()
    except Exception as e:
        log_error(errors_log_path, "EDA_PLOTS", e)

    X = df.drop(columns=["target"]).reset_index(drop=True)
    y = df["target"].reset_index(drop=True)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    folds = list(skf.split(X, y))

    # fold_records["ModelName"] = [ {fold, Accuracy, ... }, ... ]
    fold_records = {name: [] for name in MODEL_ORDER}
    # out-of-fold probabilities (kept for optional downstream analysis)
    oof_probs = {}        # name -> np.array length len(y), filled per fold
    oof_filled = {}        # name -> bool array marking which rows were filled

    def get_oof_array(name):
        if name not in oof_probs:
            oof_probs[name] = np.full(len(y), np.nan)
        return oof_probs[name]

    classical_model_ctors = {
        "LogisticRegression": lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
        "NaiveBayes": lambda: GaussianNB(),
        "RandomForest": lambda: RandomForestClassifier(
            n_estimators=500, max_depth=20, class_weight="balanced", random_state=RANDOM_STATE),
        "ExtraTrees": lambda: ExtraTreesClassifier(
            n_estimators=500, class_weight="balanced", random_state=RANDOM_STATE),
        "XGBoost": lambda: XGBClassifier(
            n_estimators=500, max_depth=8, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
            random_state=RANDOM_STATE),
        "LightGBM": lambda: LGBMClassifier(
            n_estimators=500, learning_rate=0.05, random_state=RANDOM_STATE, verbose=-1),
        "CatBoost": lambda: CatBoostClassifier(
            iterations=500, learning_rate=0.05, depth=8, verbose=0),
    }

    pt_model_config_ctors = {
        "FTTransformer": lambda: FTTransformerConfig(task="classification", learning_rate=1e-3),
        "TabTransformer": lambda: TabTransformerConfig(task="classification", learning_rate=1e-3),
        "AutoInt": lambda: AutoIntConfig(task="classification", learning_rate=1e-3),
        "GANDALF": lambda: GANDALFConfig(task="classification", learning_rate=1e-3),
        "DANet": lambda: DANetConfig(task="classification", learning_rate=1e-3),
    }
    # NODE intentionally removed.

    # ------------------------------------------------------
    # Helpers for leak-free Meta-Stacking
    # ------------------------------------------------------
    def maybe_smote(X_df, y_ser, tag):
        if EXECUTION_MODE in (2, 4):
            try:
                return BorderlineSMOTE(
                    random_state=RANDOM_STATE, k_neighbors=5
                ).fit_resample(X_df, y_ser)
            except Exception as e:
                log_error(errors_log_path, f"{tag}_SMOTE", e)
        return X_df, y_ser

    def fit_predict_base(name, X_fit_raw, y_fit_raw, eval_dfs):
        """Fit ONE base learner on (X_fit_raw, y_fit_raw) [+SMOTE in modes 2/4]
        and return a list of P(class=1) arrays, one per DataFrame in eval_dfs."""
        if name == "TabNet":
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_fit_raw, y_fit_raw, test_size=0.15,
                stratify=y_fit_raw, random_state=RANDOM_STATE)
            X_tr, y_tr = maybe_smote(X_tr, y_tr, "MetaTabNet")
            m = TabNetClassifier(
                n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=1e-4,
                optimizer_params=dict(lr=2e-2), mask_type="entmax")
            m.fit(X_tr.values.astype(np.float32), np.asarray(y_tr),
                  eval_set=[(X_val.values.astype(np.float32), np.asarray(y_val))],
                  max_epochs=100, patience=20,
                  batch_size=256, virtual_batch_size=128)
            return [m.predict_proba(d.values.astype(np.float32))[:, 1] for d in eval_dfs]

        X_fit, y_fit = maybe_smote(X_fit_raw, y_fit_raw, f"Meta_{name}")

        if name in classical_model_ctors:
            m = classical_model_ctors[name]()
            m.fit(X_fit, y_fit)
            return [m.predict_proba(d)[:, 1] for d in eval_dfs]

        # PyTorch-Tabular models
        fit_df = X_fit.copy()
        fit_df["target"] = np.asarray(y_fit)
        m = TabularModel(
            data_config=DataConfig(target=["target"], continuous_cols=list(X_fit.columns)),
            model_config=pt_model_config_ctors[name](),
            optimizer_config=OptimizerConfig(),
            trainer_config=TrainerConfig(batch_size=256, max_epochs=100),
        )
        m.fit(train=fit_df)
        outs = []
        for d in eval_dfs:
            dd = d.copy()
            dd["target"] = 0  # placeholder; not used for prediction
            pdf = m.predict(dd)
            prob_cols = sorted([c for c in pdf.columns if c.endswith("_probability")])
            outs.append(pdf[prob_cols[-1]].values)
        return outs

    # ------------------------------------------------------
    # MAIN K-FOLD LOOP
    # ------------------------------------------------------
    for fold_idx, (train_idx, test_idx) in enumerate(folds, start=1):
        print(f"\n  --- Fold {fold_idx}/{N_SPLITS} ---")
        print("\n" + "=" * 70)
        print(f"FOLD {fold_idx}")
        print("=" * 70)

        X_train_raw, X_test_raw = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
        y_train_raw, y_test_raw = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)



        # Train-fold-only imputation
        train_medians = X_train_raw.median()

        X_train_raw = X_train_raw.fillna(train_medians).fillna(0)
        X_test_raw = X_test_raw.fillna(train_medians).fillna(0)
        # ========================================================
        # FEATURE SELECTION
        #
        # IMPORTANT:
        # This happens AFTER the split.
        #
        # ONLY X_train/y_train are passed to the selector.
        #
        # NOTE: select_features_fold must be defined elsewhere (e.g. an
        # earlier Colab cell). It must use ONLY its X_train / y_train args.
        # ========================================================


        if EXECUTION_MODE in (3, 4):

            selected_features = select_features_fold(
                X_train=X_train_raw,
                y_train=y_train_raw,
                dataset_name=dataset_name,
                fold_idx=fold_idx,
                output_root=out_dir
            )

            # ----------------------------------------------------
            # Apply the SAME selected feature list to both
            # training and test data.
            #
            # Test data is NEVER used for selecting features.
            # ----------------------------------------------------

            X_train_raw = X_train_raw[
                selected_features
            ].copy()

            X_test_raw = X_test_raw[
                selected_features
            ].copy()

            feature_cols = selected_features

        else:

            # No feature selection
            feature_cols = list(
                X_train_raw.columns
            )


        # SMOTE on the TRAIN fold only (never touches test fold -> no leakage)
        if EXECUTION_MODE in (2, 4):
            try:
                smote = BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=5)
                X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)
            except Exception as e:
                log_error(errors_log_path, f"SMOTE_fold{fold_idx}", e)
                X_train, y_train = X_train_raw, y_train_raw
        else:
            X_train, y_train = X_train_raw, y_train_raw

        X_test, y_test = X_test_raw, y_test_raw

        X_train_np = X_train.values.astype(np.float32)
        X_test_np = X_test.values.astype(np.float32)
        y_train_np = np.asarray(y_train)
        y_test_np = np.asarray(y_test)

        # fold_base_probs collects predicted probabilities from PURE models only
        # (Classical / DeepTabular / Transformer). Ensemble outputs (Soft/Hard
        # Voting, sklearn Stacking) are tracked separately in fold_ensemble_probs
        # and are NEVER fed into Meta-Stacking.
        fold_base_probs = {}      # name -> y_prob on this fold's test set (pure models only)
        fold_ensemble_probs = {}  # name -> y_prob for Soft/Hard Voting / Stacking
        fitted_classical_fold = {}

        # ---- Classical / GBDT ----
        for name, ctor in classical_model_ctors.items():
            try:
                model = ctor()
                with _TimedBlock() as t_train:
                    model.fit(X_train, y_train)
                with _TimedBlock() as t_infer:
                    y_pred = model.predict(X_test)
                    y_prob = model.predict_proba(X_test)[:, 1]

                metrics = compute_metrics(y_test, y_pred, y_prob)
                metrics["fold"] = fold_idx
                metrics["train_time_sec"] = t_train.seconds
                metrics["infer_time_sec"] = t_infer.seconds
                metrics["train_gpu_mb"] = t_train.gpu_mb
                metrics["infer_gpu_mb"] = t_infer.gpu_mb
                fold_records[name].append(metrics)

                model_dir = os.path.join(out_dir, name)
                save_fold_artifacts(model_dir, name, fold_idx, y_test, y_pred, y_prob)

                fold_base_probs[name] = y_prob
                fitted_classical_fold[name] = model
                get_oof_array(name)[test_idx] = y_prob
                print(f"    [OK] {name}: ROC_AUC={metrics['ROC_AUC']:.3f} "
                      f"(train={t_train.seconds:.2f}s, infer={t_infer.seconds:.3f}s)")
            except Exception as e:
                log_error(errors_log_path, f"{name}_fold{fold_idx}", e)
                print(f"    [FAIL] {name} -- see errors.log")

        # ---- TabNet ----
        try:
            tabnet = TabNetClassifier(
                n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=1e-4,
                optimizer_params=dict(lr=2e-2), mask_type="entmax"
            )

            # LEAKAGE FIX: early-stopping validation set is carved from the
            # TRAIN fold only (never the test fold), and BEFORE SMOTE so no
            # synthetic points can land in the validation set. SMOTE (modes
            # 2/4) is then applied to the TabNet-train part only.
            X_tab_tr, X_tab_val, y_tab_tr, y_tab_val = train_test_split(
                X_train_raw, y_train_raw,
                test_size=0.15,
                stratify=y_train_raw,
                random_state=RANDOM_STATE
            )
            if EXECUTION_MODE in (2, 4):
                try:
                    X_tab_tr, y_tab_tr = BorderlineSMOTE(
                        random_state=RANDOM_STATE, k_neighbors=5
                    ).fit_resample(X_tab_tr, y_tab_tr)
                except Exception as e:
                    log_error(errors_log_path, f"TabNet_SMOTE_fold{fold_idx}", e)

            X_tab_tr_np = X_tab_tr.values.astype(np.float32)
            y_tab_tr_np = np.asarray(y_tab_tr)
            X_tab_val_np = X_tab_val.values.astype(np.float32)
            y_tab_val_np = np.asarray(y_tab_val)

            with _TimedBlock() as t_train:
                tabnet.fit(
                    X_tab_tr_np,
                    y_tab_tr_np,
                    eval_set=[(X_tab_val_np, y_tab_val_np)],
                    max_epochs=100,
                    patience=20,
                    batch_size=256,
                    virtual_batch_size=128
                )
            with _TimedBlock() as t_infer:
                y_pred = tabnet.predict(X_test_np)
                y_prob = tabnet.predict_proba(X_test_np)[:, 1]

            metrics = compute_metrics(y_test_np, y_pred, y_prob)
            metrics["fold"] = fold_idx
            metrics["train_time_sec"] = t_train.seconds
            metrics["infer_time_sec"] = t_infer.seconds
            metrics["train_gpu_mb"] = t_train.gpu_mb
            metrics["infer_gpu_mb"] = t_infer.gpu_mb
            fold_records["TabNet"].append(metrics)

            model_dir = os.path.join(out_dir, "TabNet")
            save_fold_artifacts(model_dir, "TabNet", fold_idx, y_test_np, y_pred, y_prob)
            fold_base_probs["TabNet"] = y_prob
            get_oof_array("TabNet")[test_idx] = y_prob
            print(f"    [OK] TabNet: ROC_AUC={metrics['ROC_AUC']:.3f} "
                  f"(train={t_train.seconds:.2f}s, infer={t_infer.seconds:.3f}s, "
                  f"train_gpu={t_train.gpu_mb:.1f}MB)")
        except Exception as e:
            log_error(errors_log_path, f"TabNet_fold{fold_idx}", e)
            print("    [FAIL] TabNet -- see errors.log")

        # ---- PyTorch-Tabular family (FTTransformer, TabTransformer, AutoInt, GANDALF, DANet) ----
        train_df = X_train.copy()
        train_df["target"] = np.asarray(y_train)
        test_df = X_test.copy()
        test_df["target"] = np.asarray(y_test)

        data_config = DataConfig(target=["target"], continuous_cols=feature_cols)
        trainer_config = TrainerConfig(batch_size=256, max_epochs=100)
        optimizer_config = OptimizerConfig()

        for name, cfg_ctor in pt_model_config_ctors.items():
            try:
                pt_model = TabularModel(
                    data_config=data_config,
                    model_config=cfg_ctor(),
                    optimizer_config=optimizer_config,
                    trainer_config=trainer_config,
                )
                with _TimedBlock() as t_train:
                    pt_model.fit(train=train_df)
                with _TimedBlock() as t_infer:
                    pred_df = pt_model.predict(test_df)

                pred_col = [c for c in pred_df.columns if c.endswith("_prediction")][0]
                prob_cols = sorted([c for c in pred_df.columns if c.endswith("_probability")])
                prob_col = prob_cols[-1]

                y_pred = pred_df[pred_col].values.astype(int)
                y_prob = pred_df[prob_col].values

                metrics = compute_metrics(y_test, y_pred, y_prob)
                metrics["fold"] = fold_idx
                metrics["train_time_sec"] = t_train.seconds
                metrics["infer_time_sec"] = t_infer.seconds
                metrics["train_gpu_mb"] = t_train.gpu_mb
                metrics["infer_gpu_mb"] = t_infer.gpu_mb
                fold_records[name].append(metrics)

                model_dir = os.path.join(out_dir, name)
                save_fold_artifacts(model_dir, name, fold_idx, y_test, y_pred, y_prob)
                fold_base_probs[name] = y_prob
                get_oof_array(name)[test_idx] = y_prob
                print(f"    [OK] {name}: ROC_AUC={metrics['ROC_AUC']:.3f} "
                      f"(train={t_train.seconds:.2f}s, infer={t_infer.seconds:.3f}s, "
                      f"train_gpu={t_train.gpu_mb:.1f}MB)")
            except Exception as e:
                log_error(errors_log_path, f"{name}_fold{fold_idx}", e)
                print(f"    [FAIL] {name} -- see errors.log")

        # ---- Voting / Stacking (sklearn estimators only, refit per fold) ----
        # NOTE: these are ENSEMBLES. Their outputs go into fold_ensemble_probs,
        # NOT fold_base_probs, so they are excluded from Meta-Stacking's inputs.
        sklearn_estimators = [(n, classical_model_ctors[n]()) for n in fitted_classical_fold.keys()]
        if len(sklearn_estimators) >= 2:
            try:
                soft_voting = VotingClassifier(estimators=sklearn_estimators, voting="soft")
                with _TimedBlock() as t_train:
                    soft_voting.fit(X_train, y_train)
                with _TimedBlock() as t_infer:
                    y_pred = soft_voting.predict(X_test)
                    y_prob = soft_voting.predict_proba(X_test)[:, 1]
                metrics = compute_metrics(y_test, y_pred, y_prob)
                metrics["fold"] = fold_idx
                metrics["train_time_sec"] = t_train.seconds
                metrics["infer_time_sec"] = t_infer.seconds
                metrics["train_gpu_mb"] = t_train.gpu_mb
                metrics["infer_gpu_mb"] = t_infer.gpu_mb
                fold_records["Soft Voting"].append(metrics)
                model_dir = os.path.join(out_dir, "SoftVoting")
                save_fold_artifacts(model_dir, "Soft Voting", fold_idx, y_test, y_pred, y_prob)
                fold_ensemble_probs["Soft Voting"] = y_prob
                get_oof_array("Soft Voting")[test_idx] = y_prob
                print(f"    [OK] Soft Voting: ROC_AUC={metrics['ROC_AUC']:.3f} "
                      f"(train={t_train.seconds:.2f}s, infer={t_infer.seconds:.3f}s)")
            except Exception as e:
                log_error(errors_log_path, f"SoftVoting_fold{fold_idx}", e)

            try:
                hard_voting = VotingClassifier(estimators=sklearn_estimators, voting="hard")
                with _TimedBlock() as t_train:
                    hard_voting.fit(X_train, y_train)
                with _TimedBlock() as t_infer:
                    y_pred = hard_voting.predict(X_test)
                metrics = compute_metrics(y_test, y_pred, None)
                metrics["fold"] = fold_idx
                metrics["train_time_sec"] = t_train.seconds
                metrics["infer_time_sec"] = t_infer.seconds
                metrics["train_gpu_mb"] = t_train.gpu_mb
                metrics["infer_gpu_mb"] = t_infer.gpu_mb
                fold_records["Hard Voting"].append(metrics)
                model_dir = os.path.join(out_dir, "HardVoting")
                save_fold_artifacts(model_dir, "Hard Voting", fold_idx, y_test, y_pred, None)
                print(f"    [OK] Hard Voting: F1={metrics['F1']:.3f} "
                      f"(train={t_train.seconds:.2f}s, infer={t_infer.seconds:.3f}s)")
            except Exception as e:
                log_error(errors_log_path, f"HardVoting_fold{fold_idx}", e)

            try:
                stack_model = StackingClassifier(
                    estimators=sklearn_estimators, final_estimator=LogisticRegression(), cv=5)
                with _TimedBlock() as t_train:
                    stack_model.fit(X_train, y_train)
                with _TimedBlock() as t_infer:
                    y_pred = stack_model.predict(X_test)
                    y_prob = stack_model.predict_proba(X_test)[:, 1]
                metrics = compute_metrics(y_test, y_pred, y_prob)
                metrics["fold"] = fold_idx
                metrics["train_time_sec"] = t_train.seconds
                metrics["infer_time_sec"] = t_infer.seconds
                metrics["train_gpu_mb"] = t_train.gpu_mb
                metrics["infer_gpu_mb"] = t_infer.gpu_mb
                fold_records["Stacking (sklearn)"].append(metrics)
                model_dir = os.path.join(out_dir, "Stacking_sklearn")
                save_fold_artifacts(model_dir, "Stacking (sklearn)", fold_idx, y_test, y_pred, y_prob)
                fold_ensemble_probs["Stacking (sklearn)"] = y_prob
                get_oof_array("Stacking (sklearn)")[test_idx] = y_prob
                print(f"    [OK] Stacking (sklearn): ROC_AUC={metrics['ROC_AUC']:.3f} "
                      f"(train={t_train.seconds:.2f}s, infer={t_infer.seconds:.3f}s)")
            except Exception as e:
                log_error(errors_log_path, f"Stacking_fold{fold_idx}", e)

        # ---- Meta-Stacking (leak-free, trained on TRAIN-fold OOF predictions only) ----
        # LEAKAGE FIX: the meta-model is fit on out-of-fold base predictions
        # produced by an inner CV over the TRAIN fold. Test-fold labels are never
        # used. Test-fold meta-features = average of the inner-fold base models'
        # test probabilities. Inputs are PURE base learners only (never ensembles).
        meta_bases = [n for n in fold_base_probs.keys()
                      if (META_BASE_MODELS is None or n in META_BASE_MODELS)]
        if len(meta_bases) >= 2:
            try:
                y_test_arr = np.asarray(y_test)
                y_tr_arr = np.asarray(y_train_raw)
                n_tr = len(X_train_raw)

                k = min(META_INNER_SPLITS, int(np.min(np.bincount(y_tr_arr))))
                if k < 2:
                    raise ValueError("Too few minority samples in train fold for inner CV.")
                inner_cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)

                oof = {n: np.full(n_tr, np.nan) for n in meta_bases}
                test_acc = {n: np.zeros(len(X_test)) for n in meta_bases}
                failed = set()

                # Timing = full stacking overhead (inner-CV base refits + meta fit).
                with _TimedBlock() as t_train:
                    for in_tr, in_val in inner_cv.split(X_train_raw, y_tr_arr):
                        X_in_tr = X_train_raw.iloc[in_tr].reset_index(drop=True)
                        y_in_tr = y_train_raw.iloc[in_tr].reset_index(drop=True)
                        X_in_val = X_train_raw.iloc[in_val].reset_index(drop=True)
                        for n in meta_bases:
                            if n in failed:
                                continue
                            try:
                                p_val, p_te = fit_predict_base(n, X_in_tr, y_in_tr, [X_in_val, X_test])
                                oof[n][in_val] = p_val
                                test_acc[n] += np.asarray(p_te) / k
                            except Exception as e:
                                failed.add(n)
                                log_error(errors_log_path, f"MetaBase_{n}_fold{fold_idx}", e)

                    used = [n for n in meta_bases if n not in failed]
                    if len(used) < 2:
                        raise ValueError("Fewer than 2 base learners succeeded for Meta-Stacking.")

                    meta_train = np.column_stack([oof[n] for n in used])
                    meta_test = np.column_stack([test_acc[n] for n in used])
                    meta_model = LogisticRegression(max_iter=5000)
                    meta_model.fit(meta_train, y_tr_arr)

                with _TimedBlock() as t_infer:
                    meta_prob = meta_model.predict_proba(meta_test)[:, 1]
                    meta_pred = (meta_prob >= 0.5).astype(int)

                metrics = compute_metrics(y_test_arr, meta_pred, meta_prob)
                metrics["fold"] = fold_idx
                metrics["train_time_sec"] = t_train.seconds
                metrics["infer_time_sec"] = t_infer.seconds
                metrics["train_gpu_mb"] = t_train.gpu_mb
                metrics["infer_gpu_mb"] = t_infer.gpu_mb
                fold_records["Meta-Stacking"].append(metrics)
                model_dir = os.path.join(out_dir, "MetaStacking_All")
                save_fold_artifacts(model_dir, "Meta-Stacking", fold_idx, y_test_arr, meta_pred, meta_prob)
                get_oof_array("Meta-Stacking")[test_idx] = meta_prob
                print(f"    [OK] Meta-Stacking ({len(used)} PURE bases: {used}, "
                      f"{k}-fold inner CV on TRAIN fold only, no test labels used): "
                      f"ROC_AUC={metrics['ROC_AUC']:.3f} (train={t_train.seconds:.1f}s)")
            except Exception as e:
                log_error(errors_log_path, f"MetaStacking_fold{fold_idx}", e)
                print("    [FAIL] Meta-Stacking -- see errors.log")

    # ------------------------------------------------------
    # AGGREGATE FOLD RESULTS -> SUMMARY TABLE
    # ------------------------------------------------------
    summary_rows = []
    longform_rows = []  # one row per (model, fold) -- used for stats tests
    for name in MODEL_ORDER:
        records = fold_records.get(name, [])
        if not records:
            continue
        summary = aggregate_fold_metrics(records)
        summary_rows.append({"Model": name, **summary, "n_folds": len(records)})
        for rec in records:
            longform_rows.append({
                "Dataset": dataset_name, "Model": name,
                "Family": MODEL_FAMILY.get(name, "Other"),
                "Fold": rec["fold"], "ROC_AUC": rec["ROC_AUC"], "F1": rec["F1"],
                "MCC": rec["MCC"], "Accuracy": rec["Accuracy"],
                "Precision": rec["Precision"], "Recall": rec["Recall"],
                "train_time_sec": rec.get("train_time_sec", np.nan),
                "infer_time_sec": rec.get("infer_time_sec", np.nan),
                "train_gpu_mb": rec.get("train_gpu_mb", np.nan),
                "infer_gpu_mb": rec.get("infer_gpu_mb", np.nan),
            })
            GLOBAL_FOLD_RECORDS.append(longform_rows[-1])

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(out_dir, f"{dataset_name}_kfold_summary.csv"), index=False)
    longform_df = pd.DataFrame(longform_rows)
    longform_df.to_csv(os.path.join(out_dir, f"{dataset_name}_foldwise_metrics.csv"), index=False)

    print(f"\n  K-Fold Summary for {dataset_name} (mean ± std over {N_SPLITS} folds):")
    if not summary_df.empty:
        cols_to_show = ["Model", "ROC_AUC_mean", "ROC_AUC_std", "F1_mean", "F1_std", "MCC_mean"]
        print(summary_df[cols_to_show].to_string(index=False))

        timing_cols = ["Model", "train_time_sec_mean", "infer_time_sec_mean",
                       "train_gpu_mb_mean", "infer_gpu_mb_mean"]
        timing_cols = [c for c in timing_cols if c in summary_df.columns]
        print(f"\n  Avg Time / GPU Memory per model (over {N_SPLITS} folds):")
        print(summary_df[timing_cols].to_string(index=False))

    # ---- ROC-AUC bar plot (mean ± std) ----
    try:
        if not summary_df.empty:
            plt.figure(figsize=(14, 6))
            order = [m for m in MODEL_ORDER if m in summary_df["Model"].values]
            plot_df = summary_df.set_index("Model").loc[order].reset_index()
            palette = sns.color_palette("GnBu", n_colors=len(plot_df))
            plt.bar(plot_df["Model"], plot_df["ROC_AUC_mean"],
                    yerr=plot_df["ROC_AUC_std"], capsize=4, color=palette)
            plt.xticks(rotation=45, ha="right")
            plt.ylabel("ROC-AUC (mean ± std)")
            plt.title(f"ROC-AUC Comparison ({N_SPLITS}-Fold CV) — {dataset_name}", fontweight="bold")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "roc_auc_comparison_kfold.png"), dpi=150)
            plt.close()
    except Exception as e:
        log_error(errors_log_path, "ROC_AUC_BARPLOT", e)

    # ---- Training time / GPU memory comparison plots ----
    try:
        if not summary_df.empty and "train_time_sec_mean" in summary_df.columns:
            order = [m for m in MODEL_ORDER if m in summary_df["Model"].values]
            plot_df = summary_df.set_index("Model").loc[order].reset_index()

            fig, axes = plt.subplots(1, 2, figsize=(18, 6))

            axes[0].bar(plot_df["Model"], plot_df["train_time_sec_mean"],
                        yerr=plot_df["train_time_sec_std"], capsize=4,
                        color=sns.color_palette("GnBu", n_colors=len(plot_df)))
            axes[0].set_ylabel("Training time (sec, mean ± std)")
            axes[0].set_title(f"Avg Training Time per Fold — {dataset_name}")
            axes[0].tick_params(axis="x", rotation=45)
            for label in axes[0].get_xticklabels():
                label.set_ha("right")

            axes[1].bar(plot_df["Model"], plot_df["train_gpu_mb_mean"],
                        yerr=plot_df["train_gpu_mb_std"], capsize=4,
                        color=sns.color_palette("OrRd", n_colors=len(plot_df)))
            axes[1].set_ylabel("Peak GPU memory during training (MB, mean ± std)")
            axes[1].set_title(f"Avg GPU Memory (Training) — {dataset_name}"
                               + ("" if GPU_AVAILABLE else " [No GPU detected]"))
            axes[1].tick_params(axis="x", rotation=45)
            for label in axes[1].get_xticklabels():
                label.set_ha("right")

            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "time_gpu_comparison_kfold.png"), dpi=150)
            plt.close()
    except Exception as e:
        log_error(errors_log_path, "TIME_GPU_BARPLOT", e)

    # ------------------------------------------------------
    # STATISTICAL VALIDITY -- per dataset, across model families
    # (Ensemble rows are excluded inside run_between_family_comparison)
    # ------------------------------------------------------
    run_between_family_comparison(longform_df, out_dir, metric="ROC_AUC")

    print(f"\n  Done. Results saved to: {out_dir}")
    zip_dataset_results(out_dir, dataset_name)


# ==========================================================
# STATISTICAL VALIDITY TESTS
# ==========================================================




def run_between_family_comparison(longform_df, out_dir, metric="ROC_AUC"):
    """
    Tests whether one model family (Classical, DeepTabular, or Transformer)
    outperforms another using a Linear Mixed-Effects Model (LME).

    Ensemble methods are deliberately excluded because they are constructed
    by combining predictions from the other families and therefore do not
    represent an independent modeling paradigm.

    Only STAT_FAMILIES = ["Classical", "DeepTabular", "Transformer"] are used.

    Response = the selected metric (e.g. ROC_AUC); Family is a fixed effect.

    1. Single dataset: random intercept for Dataset::Model.
    2. Multiple datasets: crossed random effects for Dataset and Model.

    CAVEAT: the 5 CV folds share ~60% of their training data, so fold-level
    scores are not fully independent; p-values from this model are likely
    optimistic. Treat them as descriptive, or aggregate per dataset first.
    """

    print("\n  ---- BETWEEN-FAMILY STATISTICAL COMPARISON ----")

    # Drop Ensemble rows -- Ensemble is not a family for this analysis.
    df = longform_df.dropna(subset=[metric]).copy()
    df = df[df["Family"].isin(STAT_FAMILIES)].copy()
    if df.empty:
        print("  [Stats] No pure-model (non-Ensemble) rows available -- skipping.")
        return

    n_families = df["Family"].nunique()
    if n_families < 2:
        print("  [Stats] Fewer than 2 (non-Ensemble) families present -- skipping between-family comparison.")
        return

    print(f"  [Stats] Mean {metric} by family (Ensemble excluded -- families considered: {STAT_FAMILIES}):")
    print(df.groupby("Family")[metric].mean().sort_values(ascending=False).to_string())


    # ---- (B) Linear Mixed-Effects Model -- the primary, more valid test ----
    if not MIXEDLM_AVAILABLE:
        print("  [Stats] statsmodels not installed; skipping mixed-effects family comparison.")
        return
    try:
        mdf = df.copy()
        mdf["Family"] = mdf["Family"].astype("category")
        n_datasets = mdf["Dataset"].nunique()

        if n_datasets >= 2:
            # CROSSED random effects: Dataset and Model as two SEPARATE variance
            # components. statsmodels MixedLM doesn't support crossed effects via
            # `groups` alone, so we use a dummy single group plus vc_formula.
            mdf["_dummy_group"] = "ALL"
            vc_formula = {"Dataset": "0 + C(Dataset)", "Model": "0 + C(Model)"}
            model = smf.mixedlm(
                f"{metric} ~ Family", data=mdf,
                groups="_dummy_group", vc_formula=vc_formula, re_formula="0"
            )
            result = model.fit()
            mode_desc = (f"CROSSED random effects across {n_datasets} datasets "
                         f"(Dataset and Model as separate variance components); "
                         f"Ensemble excluded from Family")
        else:
            # Single dataset: absorb per-model fold-correlation with a nested random intercept.
            mdf["Group"] = mdf["Dataset"].astype(str) + "::" + mdf["Model"].astype(str)
            model = smf.mixedlm(f"{metric} ~ Family", data=mdf, groups=mdf["Group"])
            result = model.fit()
            mode_desc = ("nested random intercept per Dataset::Model (only one dataset "
                         "present); Ensemble excluded from Family")

        with open(os.path.join(out_dir, f"between_family_mixedlm_{metric}_summary.txt"), "w") as f:
            f.write(f"Mode: {mode_desc}\n\n")
            f.write(result.summary().as_text())

        print(f"  [Stats] Mixed-effects family comparison fit OK -- {mode_desc} "
              f"(reference family = {mdf['Family'].cat.categories[0]}).")
        print("  [Stats] Fixed-effect coefficients (Family vs. reference), p-values:")
        print(result.summary().tables[1])

        if n_datasets >= 2 and hasattr(result, "vcomp") and len(result.vcomp) > 0:
            print("  [Stats] Variance components (how much spread comes from Dataset vs. Model):")
            for vc_name, vc_val in zip(result.model.exog_vc.names, result.vcomp):
                print(f"      {vc_name}: variance = {vc_val:.6f}")
            print(f"      Residual: variance = {result.scale:.6f}")

        print(f"  [Stats] Summary saved to between_family_mixedlm_{metric}_summary.txt")
    except Exception as e:
        print(f"  [Stats][FAIL] Mixed-effects family comparison failed: {e}")


# ==========================================================
# MAIN
# ==========================================================

if __name__ == "__main__":
    for filename in DATASET_FILES:
        full_path = os.path.join(DATA_DIR, filename)
        if not os.path.exists(full_path):
            print(f"\n!! File not found, skipping: {full_path}")
            continue
        run_pipeline_for_dataset(filename)

    # ------------------------------------------------------
    # CROSS-DATASET STATISTICAL VALIDITY (only meaningful with >1 dataset)
    # ------------------------------------------------------
    if GLOBAL_FOLD_RECORDS:
        global_df = pd.DataFrame(GLOBAL_FOLD_RECORDS)
        global_df.to_csv(os.path.join(OUTPUT_ROOT, "ALL_DATASETS_foldwise_metrics.csv"), index=False)

        print("\n" + "=" * 70)

        # Between-family comparison (does Family A outperform Family B?) --
        # Ensemble rows are excluded inside run_between_family_comparison.
        run_between_family_comparison(global_df, OUTPUT_ROOT, metric="ROC_AUC")

    print("\n" + "=" * 70)
    print("ALL DATASETS COMPLETE")
    print(f"Results root: {OUTPUT_ROOT}")
    print("=" * 70)

GPU detected: Tesla T4

[SKIP] PC1 already zipped — reloading its fold records for global stats and resuming from next dataset.
  [SKIP] Recovered 85 fold-level rows for PC1.

[SKIP] PC2 already zipped — reloading its fold records for global stats and resuming from next dataset.
  [SKIP] Recovered 85 fold-level rows for PC2.

[SKIP] PC3 already zipped — reloading its fold records for global stats and resuming from next dataset.
  [SKIP] Recovered 85 fold-level rows for PC3.

[SKIP] PC4 already zipped — reloading its fold records for global stats and resuming from next dataset.
  [SKIP] Recovered 85 fold-level rows for PC4.

[SKIP] PC5 already zipped — reloading its fold records for global stats and resuming from next dataset.
  [SKIP] Recovered 85 fold-level rows for PC5.

[SKIP] JM1 already zipped — reloading its fold records for global stats and resuming from next dataset.
  [SKIP] Recovered 85 fold-level rows for JM1.

[SKIP] KC1 already zipped — reloading its fold records for globa